In [1]:
%load_ext rpy2.ipython

In [4]:
%%bash
# Install system dependencies and pre-compiled R binary packages via Ubuntu package manager
# This avoids compilation failures and speeds up installation significantly.
echo "Installing system dependencies and pre-compiled R packages..."
sudo apt-get update -qq
sudo apt-get install -y -qq \
  r-cran-ggplot2 \
  r-cran-rpart \
  r-cran-rpart.plot \
  r-cran-randomforest \
  r-cran-proc \
  libcurl4-openssl-dev \
  libssl-dev \
  libxml2-dev


Installing system dependencies and pre-compiled R packages...
Selecting previously unselected package r-cran-proc.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../r-cran-proc_1.19.1-1.ca2404.1_amd64.deb ...
Unpacking r-cran-proc (1.19.1-1.ca2404.1) ...
Selecting previously unselected package r-cran-randomforest.
Preparing to unpack .../r-cran-randomforest_4.7-1.2-1.ca2404.1_amd64.deb ...
Unpacking r-cran-randomforest (4.7-1.2-1.ca2404.1) ...
Selecting previously unselected package r-cran-rpart.plot.
Preparing to unpack .../r-cran-rpart.plot_3.1.5-1.ca2404.1_all.deb ...
Unpacking r-cran-rpart.plot (3.1.5-1.ca2404.1) ...
Setting up r-cran-rpart.plot (3.1.5-1.ca2404.1) ...
Setting up r-cran-proc (1.19.1-1.ca2404.1) ...
Setting up r-cran-randomforest (4.7-1.2-1.ca2404.1) ...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [5]:
import rpy2.robjects as robjects
# Verify that all required libraries can be loaded cleanly into the R environment
print("Verifying R package installations...")
verification_code = """
required <- c("ggplot2", "rpart", "rpart.plot", "randomForest", "pROC")
results <- sapply(required, requireNamespace, quietly = TRUE)
print(results)
if(all(results)) {
  cat("\\nALL LIBRARIES INSTALLED AND LOADED SUCCESSFULLY!\\n")
} else {
  cat("\\nSome libraries are still missing. Attempting fallback installation...\\n")
  install.packages(names(results[!results]), repos='https://cloud.r-project.org')
}
"""
robjects.r(verification_code)

Verifying R package installations...
     ggplot2        rpart   rpart.plot randomForest         pROC 
        TRUE         TRUE         TRUE         TRUE         TRUE 

ALL LIBRARIES INSTALLED AND LOADED SUCCESSFULLY!


In [2]:
%%R
# ================================================================
# TELCO CUSTOMER CHURN CHURNGUARD (Review 1/Phase 1): SIMPLE DATA PROCESSING + EDA
# Dataset: IBM Telco Customer Churn
# Requirements: R + ggplot2 (the script installs ggplot2 if needed)
# Output folder: telco_eda_outputs
# ================================================================

# 1. Setup ----------------------------------------------------------------
required_packages <- c("ggplot2")
missing_packages <- required_packages[!required_packages %in% rownames(installed.packages())]
if (length(missing_packages) > 0) {
  install.packages(missing_packages, dependencies = NA)
}
if (!requireNamespace("ggplot2", quietly = TRUE)) {
  stop("ggplot2 could not be installed. Install it with install.packages('ggplot2') and rerun.")
}
library(ggplot2)

out_dir <- "telco_eda_outputs"
if (!dir.exists(out_dir)) dir.create(out_dir)

# 2. Load data automatically; choose a CSV only if download fails -----------
urls <- c(
  "https://raw.githubusercontent.com/SohelRaja/Customer-Churn-Analysis/master/Datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv",
  "https://raw.githubusercontent.com/MainakRepositor/Datasets/master/WA_Fn-UseC_-Telco-Customer-Churn.csv",
  "https://raw.githubusercontent.com/rstudio/keras-customer-churn/master/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)
data_file <- file.path(out_dir, "Telco-Customer-Churn.csv")
download_ok <- FALSE
for (u in urls) {
  try({
    utils::download.file(u, destfile = data_file, mode = "wb", quiet = TRUE)
    if (file.exists(data_file) && file.info(data_file)$size > 10000) download_ok <- TRUE
  }, silent = TRUE)
  if (download_ok) break
}
if (!download_ok) {
  message("Automatic download failed. Please select the Telco Customer Churn CSV file.")
  data_file <- file.choose()
}

raw_data <- read.csv(data_file, stringsAsFactors = FALSE, na.strings = c("", "NA", " "))
cat("\n--- DATASET OVERVIEW ---\n")
cat("Rows:", nrow(raw_data), " Columns:", ncol(raw_data), "\n")
cat("Column names:\n"); print(names(raw_data))
cat("\nStructure:\n"); str(raw_data)
cat("\nFirst six rows:\n"); print(head(raw_data))

# Save initial summary
capture.output(summary(raw_data), file = file.path(out_dir, "01_initial_summary.txt"))

# 3. Basic data cleaning ---------------------------------------------------
# Standardize column names to lowercase with underscores.
names(raw_data) <- tolower(gsub("[^A-Za-z0-9]+", "_", names(raw_data)))

# Convert blank TotalCharges values to NA, then convert to numeric.
if ("totalcharges" %in% names(raw_data)) {
  raw_data$totalcharges[trimws(raw_data$totalcharges) == ""] <- NA
  raw_data$totalcharges <- suppressWarnings(as.numeric(raw_data$totalcharges))
}

# Remove exact duplicate rows.
duplicates_before <- sum(duplicated(raw_data))
data <- raw_data[!duplicated(raw_data), , drop = FALSE]
cat("\nExact duplicate rows removed:", duplicates_before, "\n")

# Convert character columns to factors after missing-value handling.
# Missing values: numeric -> median; categorical -> most frequent value (mode).
get_mode <- function(x) {
  x <- x[!is.na(x) & trimws(as.character(x)) != ""]
  if (length(x) == 0) return("Unknown")
  vals <- unique(x)
  vals[which.max(tabulate(match(x, vals)))]
}

missing_before <- data.frame(
  column = names(data),
  missing_count = sapply(data, function(x) sum(is.na(x))),
  missing_percent = round(sapply(data, function(x) mean(is.na(x)) * 100), 2)
)
write.csv(missing_before, file.path(out_dir, "02_missing_values_before.csv"), row.names = FALSE)

for (nm in names(data)) {
  if (is.numeric(data[[nm]])) {
    if (anyNA(data[[nm]])) {
      med <- median(data[[nm]], na.rm = TRUE)
      if (is.finite(med)) data[[nm]][is.na(data[[nm]])] <- med
    }
  } else {
    data[[nm]] <- trimws(as.character(data[[nm]]))
    data[[nm]][data[[nm]] == ""] <- NA
    if (anyNA(data[[nm]])) {
      fill_value <- as.character(get_mode(data[[nm]]))
      data[[nm]][is.na(data[[nm]])] <- fill_value
    }
  }
}

# 4. Data-quality checks ---------------------------------------------------
cat("\nMissing values after imputation:", sum(is.na(data)), "\n")
cat("Duplicate rows after cleaning:", sum(duplicated(data)), "\n")
write.csv(data, file.path(out_dir, "03_cleaned_data.csv"), row.names = FALSE)

# 5. Outlier detection and treatment (IQR capping) -------------------------
# Detect outliers in numeric columns and cap them at Q1 - 1.5*IQR / Q3 + 1.5*IQR.
numeric_names <- names(data)[sapply(data, is.numeric)]
outlier_report <- data.frame(column = character(), outlier_count = integer(),
                             lower_bound = numeric(), upper_bound = numeric())
for (nm in numeric_names) {
  x <- data[[nm]]
  if (length(unique(x[!is.na(x)])) > 2) {
    q <- quantile(x, c(0.25, 0.75), na.rm = TRUE, names = FALSE)
    iqr_value <- q[2] - q[1]
    lo <- q[1] - 1.5 * iqr_value
    hi <- q[2] + 1.5 * iqr_value
    n_out <- sum(x < lo | x > hi, na.rm = TRUE)
    outlier_report <- rbind(outlier_report,
                            data.frame(column = nm, outlier_count = n_out, lower_bound = lo, upper_bound = hi))
    data[[nm]] <- pmin(pmax(x, lo), hi)
  }
}
write.csv(outlier_report, file.path(out_dir, "04_outlier_report.csv"), row.names = FALSE)

# 6. Transformation and feature engineering -------------------------------
# Standardize numerical columns; keep original cleaned values in data.
scaled_data <- data
for (nm in numeric_names) {
  x <- scaled_data[[nm]]
  s <- sd(x, na.rm = TRUE)
  if (is.finite(s) && s > 0) scaled_data[[nm]] <- as.numeric(scale(x))
}
write.csv(scaled_data, file.path(out_dir, "05_standardized_data.csv"), row.names = FALSE)

# Simple tenure groups for interpretable analysis.
if ("tenure" %in% names(data)) {
  data$tenure_group <- cut(data$tenure,
                           breaks = c(-Inf, 12, 24, 48, Inf),
                           labels = c("0-12 months", "13-24 months", "25-48 months", "49+ months"),
                           right = TRUE)
}

# Encode categorical variables as integer labels for a basic ML-ready file.
encoded_data <- data
for (nm in names(encoded_data)) {
  if (is.character(encoded_data[[nm]]) || is.factor(encoded_data[[nm]])) {
    encoded_data[[nm]] <- as.integer(factor(encoded_data[[nm]]))
  }
}
write.csv(encoded_data, file.path(out_dir, "06_encoded_data.csv"), row.names = FALSE)

# 7. EDA: helper to save plots --------------------------------------------
save_plot <- function(plot, filename, width = 8, height = 5) {
  ggsave(filename = file.path(out_dir, filename), plot = plot,
         width = width, height = height, dpi = 120)
}

# Univariate EDA: churn distribution
if ("churn" %in% names(data)) {
  p <- ggplot(data, aes(x = factor(churn))) +
    geom_bar() + labs(title = "Customer Churn Distribution", x = "Churn", y = "Number of customers") +
    theme_minimal()
  save_plot(p, "07_churn_distribution.png")
}

# Histograms for numeric columns
for (nm in numeric_names) {
  p <- ggplot(data, aes(x = .data[[nm]])) +
    geom_histogram(bins = 30, fill = "steelblue", color = "white") +
    labs(title = paste("Distribution of", nm), x = nm, y = "Count") + theme_minimal()
  save_plot(p, paste0("histogram_", nm, ".png"))
}

# Boxplots for numeric columns
for (nm in numeric_names) {
  p <- ggplot(data, aes(y = .data[[nm]])) +
    geom_boxplot(fill = "orange", alpha = 0.7) +
    labs(title = paste("Boxplot of", nm), y = nm) + theme_minimal()
  save_plot(p, paste0("boxplot_", nm, ".png"))
}

# Categorical frequency plots: limit to first 8 categories for readability
categorical_names <- names(data)[sapply(data, function(x) is.character(x) || is.factor(x))]
for (nm in categorical_names) {
  if (length(unique(data[[nm]])) <= 15) {
    p <- ggplot(data, aes(x = .data[[nm]])) + geom_bar(fill = "darkseagreen") +
      labs(title = paste("Category Counts:", nm), x = nm, y = "Count") +
      theme_minimal() + theme(axis.text.x = element_text(angle = 35, hjust = 1))
    save_plot(p, paste0("category_", nm, ".png"), width = 9)
  }
}

# Bivariate EDA: churn rate by categorical features
if ("churn" %in% names(data)) {
  for (nm in setdiff(categorical_names, "churn")) {
    if (length(unique(data[[nm]])) <= 15) {
      churn_summary <- aggregate(data$churn == "Yes",
                                 by = list(category = data[[nm]]), FUN = mean)
      names(churn_summary)[2] <- "churn_rate"
      p <- ggplot(churn_summary, aes(x = category, y = churn_rate)) +
        geom_col(fill = "coral") +
        scale_y_continuous(labels = function(x) paste0(round(x * 100), "%")) +
        labs(title = paste("Churn Rate by", nm), x = nm, y = "Churn rate") +
        theme_minimal() + theme(axis.text.x = element_text(angle = 35, hjust = 1))
      save_plot(p, paste0("churn_by_", nm, ".png"), width = 9)
    }
  }
}

# Numeric feature vs churn boxplots
if ("churn" %in% names(data)) {
  for (nm in numeric_names) {
    p <- ggplot(data, aes(x = factor(churn), y = .data[[nm]])) +
      geom_boxplot(fill = "plum") +
      labs(title = paste(nm, "by Churn Status"), x = "Churn", y = nm) + theme_minimal()
    save_plot(p, paste0("", nm, "_by_churn_boxplot.png"))
  }
}

# Correlation matrix and heatmap for numeric features
if (length(numeric_names) >= 2) {
  cor_mat <- cor(data[, numeric_names, drop = FALSE], use = "pairwise.complete.obs")
  write.csv(cor_mat, file.path(out_dir, "08_correlation_matrix.csv"))
  cor_df <- as.data.frame(as.table(cor_mat))
  names(cor_df) <- c("Feature1", "Feature2", "Correlation")
  p <- ggplot(cor_df, aes(x = Feature1, y = Feature2, fill = Correlation)) +
    geom_tile() + geom_text(aes(label = sprintf("%.2f", Correlation)), size = 3) +
    scale_fill_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
    labs(title = "Correlation Heatmap", x = "", y = "") +
    theme_minimal() + theme(axis.text.x = element_text(angle = 35, hjust = 1))
  save_plot(p, "09_correlation_heatmap.png", width = 8, height = 7)
}

# Multivariate EDA: pairwise scatterplots for available numeric columns
if (length(numeric_names) >= 2) {
  selected_num <- head(numeric_names, 5)
  pairs_png <- file.path(out_dir, "10_pairwise_scatterplots.png")
  png(pairs_png, width = 1000, height = 800)
  pairs(data[, selected_num, drop = FALSE], main = "Pairwise Relationships of Numeric Features")
  dev.off()
}

# Optional PCA on numeric columns (useful for dimensionality reduction)
if (length(numeric_names) >= 2) {
  pca_input <- data[, numeric_names, drop = FALSE]
  pca_input <- pca_input[, sapply(pca_input, function(x) sd(x, na.rm = TRUE) > 0), drop = FALSE]
  if (ncol(pca_input) >= 2) {
    pca_fit <- prcomp(pca_input, center = TRUE, scale. = TRUE)
    pca_scores <- as.data.frame(pca_fit$x[, 1:2, drop = FALSE])
    if ("churn" %in% names(data)) pca_scores$churn <- data$churn
    p <- ggplot(pca_scores, aes(x = PC1, y = PC2, color = if ("churn" %in% names(pca_scores)) churn else NULL)) +
      geom_point(alpha = 0.5) + labs(title = "PCA: First Two Principal Components") + theme_minimal()
    save_plot(p, "11_pca_plot.png")
    write.csv(pca_scores, file.path(out_dir, "12_pca_scores.csv"), row.names = FALSE)
    write.csv(summary(pca_fit)$importance, file.path(out_dir, "13_pca_variance_explained.csv"))
  }
}

# 8. Simple anomaly flags using IQR bounds (before capping report is saved) --
# Outlier counts are recorded in 04_outlier_report.csv. This final flag uses the
# cleaned numeric values, so it is a basic screening output rather than a diagnosis.
if (length(numeric_names) > 0) {
  data$possible_numeric_anomaly <- FALSE
  for (nm in numeric_names) {
    x <- data[[nm]]
    q <- quantile(x, c(0.25, 0.75), na.rm = TRUE, names = FALSE)
    spread <- q[2] - q[1]
    data$possible_numeric_anomaly <- data$possible_numeric_anomaly |
      (x < q[1] - 1.5 * spread | x > q[2] + 1.5 * spread)
  }
  write.csv(data, file.path(out_dir, "14_data_with_anomaly_flag.csv"), row.names = FALSE)
}

# 9. Compact report --------------------------------------------------------
report_lines <- c(
  "TELCO CUSTOMER CHURN - SIMPLE PROCESSING AND EDA REPORT",
  paste("Rows after duplicate removal:", nrow(data)),
  paste("Columns after feature engineering:", ncol(data)),
  paste("Exact duplicate rows removed:", duplicates_before),
  paste("Missing values after imputation:", sum(is.na(data))),
  "",
  "Numeric summary:",
  capture.output(summary(data[, intersect(numeric_names, names(data)), drop = FALSE]), type = "output"),
  "",
  "Outlier report (IQR bounds):",
  capture.output(print(outlier_report), type = "output"),
  "",
  "Notes:",
  "- Mean/median/mode imputation: median used for numeric columns; mode for categorical columns.",
  "- Outliers: detected and capped using the 1.5*IQR rule.",
  "- Categorical encoding: integer category codes saved separately; these codes are labels, not rankings.",
  "- Scaling: standardized numeric data saved separately.",
  "- PCA: run when at least two usable numeric columns are present.",
  "- Time-series, geospatial, and text-specific EDA are not suitable for this dataset's standard columns."
)
writeLines(report_lines, file.path(out_dir, "15_final_report.txt"))

cat("\nDONE! Processing and EDA outputs are saved in:", normalizePath(out_dir), "\n")
cat("Main files: 03_cleaned_data.csv, 04_outlier_report.csv, 05_standardized_data.csv, 06_encoded_data.csv, 08_correlation_matrix.csv, 15_final_report.txt\n")
cat("PNG charts are also saved in the same folder.\n")




--- DATASET OVERVIEW ---
Rows: 7043  Columns: 21 
Column names:
 [1] "customerID"       "gender"           "SeniorCitizen"    "Partner"         
 [5] "Dependents"       "tenure"           "PhoneService"     "MultipleLines"   
 [9] "InternetService"  "OnlineSecurity"   "OnlineBackup"     "DeviceProtection"
[13] "TechSupport"      "StreamingTV"      "StreamingMovies"  "Contract"        
[17] "PaperlessBilling" "PaymentMethod"    "MonthlyCharges"   "TotalCharges"    
[21] "Churn"           

Structure:
'data.frame':	7043 obs. of  21 variables:
 $ customerID      : chr  "7590-VHVEG" "5575-GNVDE" "3668-QPYBK" "7795-CFOCW" ...
 $ gender          : chr  "Female" "Male" "Male" "Male" ...
 $ SeniorCitizen   : int  0 0 0 0 0 0 0 0 0 0 ...
 $ Partner         : chr  "Yes" "No" "No" "No" ...
 $ Dependents      : chr  "No" "No" "No" "No" ...
 $ tenure          : int  1 34 2 45 2 8 22 10 28 62 ...
 $ PhoneService    : chr  "No" "Yes" "Yes" "No" ...
 $ MultipleLines   : chr  "No phone service" "No" "

In [7]:

%%R
# ================================================================
# CHURNGUARD - REPORT 2: IMPROVED MODEL TRAINING AND EVALUATION
# Dataset: IBM Telco Customer Churn
# IMPORTANT: Phase 1 remains unchanged.
#
# Models:
#   1. Majority-class baseline
#   2. Logistic Regression
#   3. Decision Tree (CART)
#   4. Random Forest
#
# Improvements:
#   - Stratified train/test split before statistical preprocessing
#   - Training-only imputation and factor-level handling
#   - Accuracy, precision, recall, specificity, F1, balanced accuracy
#   - ROC-AUC and Brier score
#   - Threshold sensitivity analysis
#   - Confusion matrices, ROC curves and feature importance
#   - Saved models, predictions, audit files and output manifest
# ================================================================

# 0. Setup ---------------------------------------------------------

required_packages <- c(
  "ggplot2", "rpart", "rpart.plot", "randomForest", "pROC"
)

missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages) > 0) {
  install.packages(missing_packages, dependencies = TRUE)
}

invisible(lapply(required_packages, library, character.only = TRUE))

set.seed(20261026)

output_dir <- "churn_guard_report2_outputs"
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

# 1. Load original dataset ----------------------------------------

# Prefer the source file used by Phase 1 if it exists.
candidate_paths <- c(
  if (exists("data_file") && length(data_file) > 0) {
    as.character(data_file)
  } else {
    character(0)
  },
  "/content/WA_Fn-UseC_-Telco-Customer-Churn.csv",
  "/content/WA_Fn-UseC_-Telco-Customer-Churn (1).csv",
  "WA_Fn-UseC_-Telco-Customer-Churn.csv",
  file.path("telco_eda_outputs", "Telco-Customer-Churn.csv")
)

candidate_paths <- unique(candidate_paths)
existing_paths <- candidate_paths[file.exists(candidate_paths)]

data_path <- if (length(existing_paths) > 0) {
  existing_paths[1]
} else {
  NA_character_
}

# Try public mirrors only if the dataset is not available locally.
if (is.na(data_path)) {
  urls <- c(
    "https://raw.githubusercontent.com/SohelRaja/Customer-Churn-Analysis/master/Datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/MainakRepositor/Datasets/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
  )

  for (u in urls) {
    tmp <- file.path(
      output_dir,
      "WA_Fn-UseC_-Telco-Customer-Churn.csv"
    )

    success <- tryCatch({
      utils::download.file(
        u, destfile = tmp, mode = "wb", quiet = TRUE
      )
      file.exists(tmp) && file.info(tmp)$size > 10000
    }, error = function(e) FALSE)

    if (isTRUE(success)) {
      data_path <- tmp
      break
    }
  }
}

if (is.na(data_path)) {
  stop(
    "Dataset not found. Upload the original Telco Customer Churn ",
    "CSV to /content in Google Colab and rerun this cell."
  )
}

cat("Dataset used:", data_path, "\n")

raw <- read.csv(
  data_path,
  stringsAsFactors = FALSE,
  na.strings = c("", "NA", " ")
)

# Standardise column names for consistent formulas.
names(raw) <- tolower(gsub("[^A-Za-z0-9]+", "_", names(raw)))

required_columns <- c("churn", "tenure", "totalcharges")
missing_columns <- setdiff(required_columns, names(raw))

if (length(missing_columns) > 0) {
  stop(
    "Required column(s) missing: ",
    paste(missing_columns, collapse = ", ")
  )
}

# Convert numeric columns safely.
raw$totalcharges <- suppressWarnings(
  as.numeric(trimws(as.character(raw$totalcharges)))
)
raw$tenure <- suppressWarnings(as.numeric(as.character(raw$tenure)))

if ("monthlycharges" %in% names(raw)) {
  raw$monthlycharges <- suppressWarnings(
    as.numeric(as.character(raw$monthlycharges))
  )
}

if ("seniorcitizen" %in% names(raw)) {
  raw$seniorcitizen <- suppressWarnings(
    as.numeric(as.character(raw$seniorcitizen))
  )
}

# Audit and remove exact duplicates.
n_raw <- nrow(raw)
n_columns <- ncol(raw)
duplicate_count <- sum(duplicated(raw))

data <- raw[!duplicated(raw), , drop = FALSE]

if (any(!data$churn %in% c("No", "Yes"), na.rm = TRUE)) {
  stop("Unexpected Churn values. Inspect unique(data$churn).")
}

if (anyNA(data$churn)) {
  stop("The target column contains missing values. Resolve these first.")
}

data$churn <- factor(data$churn, levels = c("No", "Yes"))

initial_audit <- data.frame(
  item = c(
    "Raw rows",
    "Raw columns",
    "Exact duplicate rows removed",
    "Missing cells before imputation",
    "Customers retained",
    "Customers churned",
    "Churn rate percent"
  ),
  value = c(
    n_raw,
    n_columns,
    duplicate_count,
    sum(is.na(data)),
    sum(data$churn == "No"),
    sum(data$churn == "Yes"),
    round(mean(data$churn == "Yes") * 100, 4)
  )
)

write.csv(
  initial_audit,
  file.path(output_dir, "01_initial_data_audit.csv"),
  row.names = FALSE
)

missing_audit <- data.frame(
  column = names(data),
  missing_count = vapply(data, function(x) sum(is.na(x)), numeric(1)),
  missing_percent = vapply(
    data, function(x) mean(is.na(x)) * 100, numeric(1)
  )
)

write.csv(
  missing_audit,
  file.path(output_dir, "02_missing_values_before_split.csv"),
  row.names = FALSE
)

cat("\n--- INITIAL DATA AUDIT ---\n")
print(initial_audit)

cat("\n--- COLUMNS WITH MISSING VALUES ---\n")
print(missing_audit[missing_audit$missing_count > 0, , drop = FALSE])

# 2. Stratified train/test split ----------------------------------

# The split happens before learning imputation values or model parameters.
set.seed(20261026)

train_indices <- unlist(lapply(levels(data$churn), function(class_name) {
  class_indices <- which(data$churn == class_name)
  sample(
    class_indices,
    size = max(1, floor(0.80 * length(class_indices)))
  )
}), use.names = FALSE)

train_indices <- sort(unique(train_indices))
test_indices <- setdiff(seq_len(nrow(data)), train_indices)

train_raw <- data[train_indices, , drop = FALSE]
test_raw <- data[test_indices, , drop = FALSE]

split_summary <- data.frame(
  split = c("Training", "Testing", "Full cleaned dataset"),
  rows = c(nrow(train_raw), nrow(test_raw), nrow(data)),
  churn_no = c(
    sum(train_raw$churn == "No"),
    sum(test_raw$churn == "No"),
    sum(data$churn == "No")
  ),
  churn_yes = c(
    sum(train_raw$churn == "Yes"),
    sum(test_raw$churn == "Yes"),
    sum(data$churn == "Yes")
  ),
  churn_rate_percent = round(
    c(
      mean(train_raw$churn == "Yes"),
      mean(test_raw$churn == "Yes"),
      mean(data$churn == "Yes")
    ) * 100,
    4
  )
)

write.csv(
  split_summary,
  file.path(output_dir, "03_train_test_split_summary.csv"),
  row.names = FALSE
)

cat("\n--- TRAIN/TEST SPLIT ---\n")
print(split_summary)

# 3. Training-only preprocessing ----------------------------------

# CustomerID is an identifier and is excluded from model predictors.
predictor_names <- setdiff(names(data), c("churn", "customerid"))

train_x <- train_raw[, predictor_names, drop = FALSE]
test_x <- test_raw[, predictor_names, drop = FALSE]

# Domain-informed rule:
# Blank TotalCharges with tenure == 0 is treated as zero.
domain_train <- is.na(train_x$totalcharges) & train_x$tenure == 0
domain_test <- is.na(test_x$totalcharges) & test_x$tenure == 0

train_x$totalcharges[domain_train] <- 0
test_x$totalcharges[domain_test] <- 0

imputation_log <- data.frame(
  column = character(),
  method = character(),
  replacement = character(),
  training_missing_before = integer(),
  testing_missing_before = integer(),
  stringsAsFactors = FALSE
)

add_log <- function(column, method, replacement, n_train, n_test) {
  imputation_log <<- rbind(
    imputation_log,
    data.frame(
      column = column,
      method = method,
      replacement = as.character(replacement),
      training_missing_before = as.integer(n_train),
      testing_missing_before = as.integer(n_test),
      stringsAsFactors = FALSE
    )
  )
}

if (sum(domain_train) + sum(domain_test) > 0) {
  add_log(
    "totalcharges",
    "Domain-informed rule: tenure equals zero",
    0,
    sum(domain_train),
    sum(domain_test)
  )
}

# Deterministic mode: first occurring category wins ties.
mode_value <- function(x) {
  x <- as.character(x)
  x <- trimws(x)
  x <- x[!is.na(x) & x != ""]

  if (length(x) == 0) return("Unknown")

  values <- unique(x)
  values[which.max(tabulate(match(x, values)))]
}

numeric_predictors <- names(train_x)[
  vapply(train_x, is.numeric, logical(1))
]

categorical_predictors <- setdiff(
  names(train_x), numeric_predictors
)

# Numeric missing values: training-set median only.
for (nm in numeric_predictors) {
  ntr <- sum(is.na(train_x[[nm]]))
  nte <- sum(is.na(test_x[[nm]]))

  median_value <- median(train_x[[nm]], na.rm = TRUE)

  if (!is.finite(median_value)) {
    median_value <- 0
  }

  if (ntr > 0 || nte > 0) {
    train_x[[nm]][is.na(train_x[[nm]])] <- median_value
    test_x[[nm]][is.na(test_x[[nm]])] <- median_value

    add_log(
      nm, "Training-set median", median_value, ntr, nte
    )
  }
}

# Categorical missing values: training-set mode only.
for (nm in categorical_predictors) {
  train_x[[nm]] <- trimws(as.character(train_x[[nm]]))
  test_x[[nm]] <- trimws(as.character(test_x[[nm]]))

  train_x[[nm]][train_x[[nm]] == ""] <- NA
  test_x[[nm]][test_x[[nm]] == ""] <- NA

  ntr <- sum(is.na(train_x[[nm]]))
  nte <- sum(is.na(test_x[[nm]]))

  fill_value <- mode_value(train_x[[nm]])

  if (ntr > 0 || nte > 0) {
    train_x[[nm]][is.na(train_x[[nm]])] <- fill_value
    test_x[[nm]][is.na(test_x[[nm]])] <- fill_value

    add_log(
      nm, "Training-set mode", fill_value, ntr, nte
    )
  }
}

# Establish categorical levels from training data only.
# Previously unseen test categories are assigned to "Other".
for (nm in categorical_predictors) {
  tr <- as.character(train_x[[nm]])
  te <- as.character(test_x[[nm]])

  train_levels <- sort(unique(tr))

  if (!("Other" %in% train_levels)) {
    train_levels <- c(train_levels, "Other")
  }

  te[!te %in% train_levels] <- "Other"

  train_x[[nm]] <- factor(tr, levels = train_levels)
  test_x[[nm]] <- factor(te, levels = train_levels)
}

if (sum(is.na(train_x)) > 0 || sum(is.na(test_x)) > 0) {
  stop("Missing predictor values remain after preprocessing.")
}

write.csv(
  imputation_log,
  file.path(output_dir, "04_training_based_imputation_log.csv"),
  row.names = FALSE
)

train_model <- data.frame(
  churn = train_raw$churn,
  train_x,
  check.names = FALSE
)

test_model <- data.frame(
  churn = test_raw$churn,
  test_x,
  check.names = FALSE
)

train_model$churn <- factor(train_model$churn, levels = c("No", "Yes"))
test_model$churn <- factor(test_model$churn, levels = c("No", "Yes"))

# Logistic regression uses a 0/1 response.
train_model$churn_yes <- as.integer(train_model$churn == "Yes")
test_model$churn_yes <- as.integer(test_model$churn == "Yes")

write.csv(
  train_model,
  file.path(output_dir, "05_training_data_prepared.csv"),
  row.names = FALSE
)

write.csv(
  test_model,
  file.path(output_dir, "06_testing_data_prepared.csv"),
  row.names = FALSE
)

# 4. Model 1: Logistic Regression ----------------------------------

logit_formula <- as.formula(
  paste("churn_yes ~", paste(predictor_names, collapse = " + "))
)

logistic_model <- glm(
  logit_formula,
  data = train_model,
  family = binomial(link = "logit"),
  control = glm.control(maxit = 100)
)

logit_prob <- as.numeric(
  predict(logistic_model, newdata = test_model, type = "response")
)

logit_prob <- pmin(pmax(logit_prob, 0), 1)

saveRDS(
  logistic_model,
  file.path(output_dir, "07_logistic_regression_model.rds")
)

capture.output(
  summary(logistic_model),
  file = file.path(output_dir, "07_logistic_regression_summary.txt")
)

# Export logistic coefficients and odds ratios.
coef_table <- as.data.frame(summary(logistic_model)$coefficients)
coef_table$term <- rownames(coef_table)
rownames(coef_table) <- NULL
names(coef_table) <- make.names(names(coef_table))

estimate_column <- which(names(coef_table) == "Estimate")[1]

if (!is.na(estimate_column)) {
  coef_table$odds_ratio <- exp(coef_table[[estimate_column]])
}

write.csv(
  coef_table,
  file.path(output_dir, "08_logistic_coefficients_odds_ratios.csv"),
  row.names = FALSE
)

# 5. Model 2: Decision Tree ----------------------------------------

tree_formula <- as.formula(
  paste("churn ~", paste(predictor_names, collapse = " + "))
)

tree_model <- rpart::rpart(
  tree_formula,
  data = train_model,
  method = "class",
  control = rpart::rpart.control(
    cp = 0.005,
    minsplit = 30,
    maxdepth = 8
  )
)

tree_prob_matrix <- predict(
  tree_model, newdata = test_model, type = "prob"
)

tree_prob <- if ("Yes" %in% colnames(tree_prob_matrix)) {
  as.numeric(tree_prob_matrix[, "Yes"])
} else {
  rep(0, nrow(test_model))
}

saveRDS(
  tree_model,
  file.path(output_dir, "09_decision_tree_model.rds")
)

capture.output(
  printcp(tree_model),
  file = file.path(output_dir, "09_decision_tree_complexity.txt")
)

# 6. Model 3: Random Forest ----------------------------------------

set.seed(20261026)

rf_model <- randomForest::randomForest(
  tree_formula,
  data = train_model[, c("churn", predictor_names), drop = FALSE],
  ntree = 500,
  importance = TRUE,
  na.action = na.omit
)

rf_prob_matrix <- predict(
  rf_model, newdata = test_model, type = "prob"
)

rf_prob <- if ("Yes" %in% colnames(rf_prob_matrix)) {
  as.numeric(rf_prob_matrix[, "Yes"])
} else {
  rep(0, nrow(test_model))
}

saveRDS(
  rf_model,
  file.path(output_dir, "10_random_forest_model.rds")
)

capture.output(
  print(rf_model),
  file = file.path(output_dir, "10_random_forest_summary.txt")
)

rf_importance <- as.data.frame(
  randomForest::importance(rf_model)
)

rf_importance$feature <- rownames(rf_importance)
rownames(rf_importance) <- NULL

rf_importance <- rf_importance[
  order(rf_importance$MeanDecreaseGini, decreasing = TRUE),
  ,
  drop = FALSE
]

write.csv(
  rf_importance,
  file.path(output_dir, "11_random_forest_feature_importance.csv"),
  row.names = FALSE
)

# 7. Evaluation metrics --------------------------------------------

# Positive class: Churn == "Yes".
# Default classification threshold: 0.50.
calculate_metrics <- function(actual, prob, model_name, threshold = 0.50) {
  actual <- factor(actual, levels = c("No", "Yes"))
  pred <- factor(
    ifelse(prob >= threshold, "Yes", "No"),
    levels = c("No", "Yes")
  )

  tn <- sum(actual == "No" & pred == "No")
  fp <- sum(actual == "No" & pred == "Yes")
  fn <- sum(actual == "Yes" & pred == "No")
  tp <- sum(actual == "Yes" & pred == "Yes")

  accuracy <- (tp + tn) / max(1, tp + tn + fp + fn)
  precision <- if (tp + fp == 0) 0 else tp / (tp + fp)
  recall <- if (tp + fn == 0) 0 else tp / (tp + fn)
  specificity <- if (tn + fp == 0) 0 else tn / (tn + fp)

  f1 <- if (precision + recall == 0) {
    0
  } else {
    2 * precision * recall / (precision + recall)
  }

  balanced_accuracy <- (recall + specificity) / 2

  roc_auc <- tryCatch({
    as.numeric(
      pROC::auc(
        pROC::roc(
          actual, prob,
          levels = c("No", "Yes"),
          direction = "<",
          quiet = TRUE
        )
      )
    )
  }, error = function(e) NA_real_)

  brier_score <- mean(
    (prob - as.integer(actual == "Yes"))^2
  )

  data.frame(
    model = model_name,
    threshold = threshold,
    accuracy = accuracy,
    precision = precision,
    recall = recall,
    specificity = specificity,
    f1_score = f1,
    balanced_accuracy = balanced_accuracy,
    roc_auc = roc_auc,
    brier_score = brier_score,
    true_negative = tn,
    false_positive = fp,
    false_negative = fn,
    true_positive = tp,
    stringsAsFactors = FALSE
  )
}

# Baseline: always predict the majority class in training data.
majority_class <- names(which.max(table(train_model$churn)))

baseline_prob <- if (majority_class == "Yes") {
  rep(1, nrow(test_model))
} else {
  rep(0, nrow(test_model))
}

probabilities <- list(
  "Majority-class baseline" = baseline_prob,
  "Logistic Regression" = logit_prob,
  "Decision Tree" = tree_prob,
  "Random Forest" = rf_prob
)

metrics <- do.call(
  rbind,
  lapply(names(probabilities), function(model_name) {
    calculate_metrics(
      test_model$churn,
      probabilities[[model_name]],
      model_name,
      threshold = 0.50
    )
  })
)

write.csv(
  metrics,
  file.path(output_dir, "12_model_evaluation_metrics.csv"),
  row.names = FALSE
)

cat("\n--- TEST-SET MODEL EVALUATION ---\n")
print(metrics, digits = 4)

# 8. Predictions and confusion matrices ----------------------------

for (model_name in names(probabilities)) {
  prob <- probabilities[[model_name]]

  pred <- factor(
    ifelse(prob >= 0.50, "Yes", "No"),
    levels = c("No", "Yes")
  )

  actual <- factor(test_model$churn, levels = c("No", "Yes"))

  cm <- as.data.frame.matrix(
    table(Actual = actual, Predicted = pred)
  )

  cm$Actual <- rownames(cm)
  cm <- cm[, c("Actual", setdiff(names(cm), "Actual")), drop = FALSE]

  safe_name <- gsub(
    "[^A-Za-z0-9]+", "_", tolower(model_name)
  )

  write.csv(
    cm,
    file.path(output_dir, paste0("confusion_matrix_", safe_name, ".csv")),
    row.names = FALSE
  )

  prediction_output <- data.frame(
    row_index_in_cleaned_data = test_indices,
    actual_churn = as.character(actual),
    predicted_churn = as.character(pred),
    predicted_churn_probability = prob,
    model = model_name,
    stringsAsFactors = FALSE
  )

  write.csv(
    prediction_output,
    file.path(output_dir, paste0("predictions_", safe_name, ".csv")),
    row.names = FALSE
  )
}

# 9. Threshold sensitivity analysis --------------------------------

thresholds <- c(0.30, 0.40, 0.50, 0.60, 0.70)

threshold_results <- do.call(
  rbind,
  lapply(
    c("Logistic Regression", "Decision Tree", "Random Forest"),
    function(model_name) {
      do.call(
        rbind,
        lapply(thresholds, function(threshold_value) {
          calculate_metrics(
            test_model$churn,
            probabilities[[model_name]],
            model_name,
            threshold = threshold_value
          )
        })
      )
    }
  )
)

write.csv(
  threshold_results,
  file.path(output_dir, "13_threshold_sensitivity_analysis.csv"),
  row.names = FALSE
)

# Important: do not choose a final deployment threshold using the test
# set alone. Use validation data or cross-validation for threshold selection.

p_threshold <- ggplot(
  threshold_results,
  aes(
    x = threshold,
    y = f1_score,
    colour = model,
    group = model
  )
) +
  geom_line(linewidth = 0.9) +
  geom_point(size = 2) +
  scale_x_continuous(breaks = thresholds) +
  coord_cartesian(ylim = c(0, 1)) +
  labs(
    title = "Threshold Sensitivity: F1-score",
    subtitle = "Exploratory test-set comparison; not for final threshold selection",
    x = "Classification threshold",
    y = "F1-score",
    colour = "Model"
  ) +
  theme_minimal(base_size = 12)

ggsave(
  file.path(output_dir, "14_threshold_sensitivity.png"),
  p_threshold,
  width = 9,
  height = 6,
  dpi = 300
)

# 10. Visualisations ------------------------------------------------

# 10a. Compare key metrics.
metrics_long <- reshape(
  metrics[
    ,
    c(
      "model", "accuracy", "precision", "recall",
      "f1_score", "balanced_accuracy", "roc_auc"
    )
  ],
  varying = c(
    "accuracy", "precision", "recall",
    "f1_score", "balanced_accuracy", "roc_auc"
  ),
  v.names = "score",
  timevar = "metric",
  times = c(
    "Accuracy", "Precision", "Recall",
    "F1-score", "Balanced accuracy", "ROC-AUC"
  ),
  direction = "long"
)

p_metrics <- ggplot(
  metrics_long,
  aes(x = metric, y = score, fill = model)
) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  coord_cartesian(ylim = c(0, 1)) +
  labs(
    title = "ChurnGuard: Model Comparison",
    subtitle = "Held-out test set; classification threshold = 0.50",
    x = "Metric",
    y = "Score",
    fill = "Model"
  ) +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 20, hjust = 1))

ggsave(
  file.path(output_dir, "15_model_metric_comparison.png"),
  p_metrics,
  width = 11,
  height = 6,
  dpi = 300
)

# 10b. ROC curves.
roc_objects <- lapply(
  c("Logistic Regression", "Decision Tree", "Random Forest"),
  function(model_name) {
    tryCatch(
      pROC::roc(
        test_model$churn,
        probabilities[[model_name]],
        levels = c("No", "Yes"),
        direction = "<",
        quiet = TRUE
      ),
      error = function(e) NULL
    )
  }
)

names(roc_objects) <- c(
  "Logistic Regression", "Decision Tree", "Random Forest"
)

roc_data <- do.call(
  rbind,
  lapply(names(roc_objects), function(model_name) {
    roc_obj <- roc_objects[[model_name]]

    if (is.null(roc_obj)) return(NULL)

    data.frame(
      false_positive_rate = 1 - roc_obj$specificities,
      true_positive_rate = roc_obj$sensitivities,
      model = model_name
    )
  })
)

if (!is.null(roc_data) && nrow(roc_data) > 0) {
  p_roc <- ggplot(
    roc_data,
    aes(
      x = false_positive_rate,
      y = true_positive_rate,
      colour = model
    )
  ) +
    geom_line(linewidth = 1) +
    geom_abline(
      intercept = 0, slope = 1,
      linetype = "dashed", colour = "grey50"
    ) +
    coord_equal() +
    labs(
      title = "ROC Curves: Held-Out Test Set",
      x = "False Positive Rate",
      y = "True Signal Rate",
      colour = "Model"
    ) +
    theme_minimal(base_size = 12)

  ggsave(
    file.path(output_dir, "16_roc_curves.png"),
    p_roc,
    width = 8,
    height = 6,
    dpi = 300
  )
}

# 10c. Decision tree plot.
png(
  file.path(output_dir, "17_decision_tree_plot.png"),
  width = 1600,
  height = 1000,
  res = 160
)

rpart.plot::rpart.plot(
  tree_model,
  type = 3,
  extra = 104,
  fallen.leaves = TRUE,
  main = "ChurnGuard Decision Tree"
)

dev.off()

# 10d. Random Forest feature importance.
importance_metric <- if (
  "MeanDecreaseGini" %in% names(rf_importance)
) {
  "MeanDecreaseGini"
} else {
  setdiff(names(rf_importance), "feature")[1]
}

importance_plot_data <- head(rf_importance, 15)

p_importance <- ggplot(
  importance_plot_data,
  aes(
    x = reorder(feature, .data[[importance_metric]]),
    y = .data[[importance_metric]]
  )
) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Random Forest: Top 15 Features",
    x = "Feature",
    y = importance_metric
  ) +
  theme_minimal(base_size = 12)

ggsave(
  file.path(output_dir, "18_random_forest_feature_importance.png"),
  p_importance,
  width = 9,
  height = 7,
  dpi = 300
)

# 10e. Confusion matrix heatmaps for fitted models.
for (model_name in c(
  "Logistic Regression", "Decision Tree", "Random Forest"
)) {
  prob <- probabilities[[model_name]]

  pred <- factor(
    ifelse(prob >= 0.50, "Yes", "No"),
    levels = c("No", "Yes")
  )

  cm_long <- as.data.frame(
    table(
      Actual = factor(test_model$churn, levels = c("No", "Yes")),
      Predicted = factor(pred, levels = c("No", "Yes"))
    )
  )

  p_cm <- ggplot(
    cm_long,
    aes(x = Predicted, y = Actual, fill = Freq)
  ) +
    geom_tile(colour = "white") +
    geom_text(aes(label = Freq), size = 5) +
    labs(
      title = paste("Confusion Matrix:", model_name),
      x = "Predicted class",
      y = "Actual class",
      fill = "Count"
    ) +
    theme_minimal(base_size = 12)

  safe_name <- gsub(
    "[^A-Za-z0-9]+", "_", tolower(model_name)
  )

  ggsave(
    file.path(
      output_dir,
      paste0("19_confusion_matrix_", safe_name, ".png")
    ),
    p_cm,
    width = 6,
    height = 5,
    dpi = 300
  )
}

# 11. Save a reusable model bundle -------------------------------

model_bundle <- list(
  logistic_regression = logistic_model,
  decision_tree = tree_model,
  random_forest = rf_model,
  predictor_names = predictor_names,
  numeric_predictors = numeric_predictors,
  categorical_predictors = categorical_predictors,
  training_factor_levels = lapply(
    train_x[categorical_predictors],
    levels
  ),
  training_majority_class = majority_class,
  default_threshold = 0.50,
  dataset_source = data_path,
  random_seed = 20261026
)

saveRDS(
  model_bundle,
  file.path(output_dir, "20_ChurnGuard_Model_Bundle.rds")
)

# 12. Final summary and output manifest ----------------------------

best_f1_row <- metrics[
  which.max(metrics$f1_score),
  ,
  drop = FALSE
]

best_auc_row <- metrics[
  which.max(metrics$roc_auc),
  ,
  drop = FALSE
]

final_summary <- c(
  "CHURNGUARD - REPORT 2 MODEL EVALUATION SUMMARY",
  paste("Dataset source:", data_path),
  paste("Raw rows:", n_raw),
  paste("Raw columns:", n_columns),
  paste("Exact duplicates removed:", duplicate_count),
  paste("Cleaned rows:", nrow(data)),
  paste("Training rows:", nrow(train_model)),
  paste("Testing rows:", nrow(test_model)),
  paste(
    "Training churn rate (%):",
    round(mean(train_model$churn == "Yes") * 100, 2)
  ),
  paste(
    "Testing churn rate (%):",
    round(mean(test_model$churn == "Yes") * 100, 2)
  ),
  "",
  "Evaluation metrics are saved in 12_model_evaluation_metrics.csv.",
  capture.output(print(metrics, digits = 4)),
  "",
  paste(
    "Highest observed F1 model:",
    best_f1_row$model,
    "| F1:",
    round(best_f1_row$f1_score, 4)
  ),
  paste(
    "Highest observed ROC-AUC model:",
    best_auc_row$model,
    "| ROC-AUC:",
    round(best_auc_row$roc_auc, 4)
  ),
  "",
  "Interpretation notes:",
  "- Accuracy alone can be misleading when churn classes are imbalanced.",
  "- Precision measures how many predicted churners actually churned.",
  "- Recall measures how many actual churners were detected.",
  "- Balanced accuracy averages sensitivity and specificity.",
  "- ROC-AUC evaluates ranking discrimination across thresholds.",
  "- Brier score measures squared probability prediction error; lower is better.",
  "- Threshold analysis is exploratory and does not select a final deployment threshold.",
  "- The test set is held out for final evaluation, not model fitting."
)

writeLines(
  final_summary,
  file.path(output_dir, "21_final_model_evaluation_summary.txt")
)

# Manifest lists files actually generated by this cell.
output_files <- list.files(
  output_dir,
  full.names = TRUE,
  recursive = TRUE
)

manifest <- data.frame(
  file_name = basename(output_files),
  relative_path = output_files,
  file_size_bytes = file.info(output_files)$size,
  generated = file.exists(output_files),
  stringsAsFactors = FALSE
)

write.csv(
  manifest,
  file.path(output_dir, "22_output_manifest.csv"),
  row.names = FALSE
)

cat("\n====================================================\n")
cat("CHURNGUARD PHASE 2 COMPLETE\n")
cat("Phase 1 was not modified by this cell.\n")
cat("Output folder:", normalizePath(output_dir), "\n")
cat("Best observed F1 model:", best_f1_row$model, "\n")
cat("Best observed ROC-AUC model:", best_auc_row$model, "\n")
cat("Review the saved metrics before making performance claims.\n")
cat("====================================================\n")


Dataset used: telco_eda_outputs/Telco-Customer-Churn.csv 

--- INITIAL DATA AUDIT ---
                             item    value
1                        Raw rows 7043.000
2                     Raw columns   21.000
3    Exact duplicate rows removed    0.000
4 Missing cells before imputation   11.000
5              Customers retained 5174.000
6               Customers churned 1869.000
7              Churn rate percent   26.537

--- COLUMNS WITH MISSING VALUES ---
                   column missing_count missing_percent
totalcharges totalcharges            11       0.1561834

--- TRAIN/TEST SPLIT ---
                 split rows churn_no churn_yes churn_rate_percent
1             Training 5634     4139      1495            26.5353
2              Testing 1409     1035       374            26.5436
3 Full cleaned dataset 7043     5174      1869            26.5370

--- TEST-SET MODEL EVALUATION ---
                    model threshold accuracy precision recall specificity
1 Majority-class basel

In [11]:
# ================================================================
# CHURNGUARD PHASE 3: COMPLETE INTERACTIVE DATA MINING APPLICATION
# Course: BCSE208P - Data Mining Lab
# Dataset: IBM Telco Customer Churn
# UI: R Shiny
#
# PROFESSOR'S REPORT 3 REQUIREMENTS:
# 1. Interactive UI design
# 2. Integration of trained data-mining models with the UI
# 3. Interactive output and visualisation
# 4. Complete project documentation and downloadable outputs
# 5. Support for live demonstration/viva
#
# PREREQUISITES:
# Run Phase 1 and the improved Phase 2 first.
# This cell does NOT modify Phase 1 or Phase 2.
# ================================================================

import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

APP_DIR = Path("/content/churn_guard_phase3")
MODEL_DIR = Path("/content/churn_guard_report2_outputs")
APP_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 1. Check Phase 2 model files
# ------------------------------------------------

required_files = [
    MODEL_DIR / "20_ChurnGuard_Model_Bundle.rds",
    MODEL_DIR / "05_training_data_prepared.csv",
    MODEL_DIR / "12_model_evaluation_metrics.csv",
    MODEL_DIR / "11_random_forest_feature_importance.csv",
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Phase 2 outputs are missing. Run Phase 2 first.\nMissing files:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------
# 2. Find R installation
# ------------------------------------------------

rscript = shutil.which("Rscript")

if rscript is None:
    raise RuntimeError(
        "Rscript was not found in this Colab runtime. "
        "Install/configure R and rpy2 first, then rerun this cell."
    )

# ------------------------------------------------
# 3. Write the complete R Shiny application
# ------------------------------------------------

app_code = r'''
options(shiny.maxRequestSize = 30 * 1024^2)

required_packages <- c(
  "shiny", "ggplot2", "randomForest", "rpart", "rpart.plot"
)

missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]

if (length(missing_packages)) {
  install.packages(
    missing_packages,
    repos = "https://cloud.r-project.org"
  )
}

library(shiny)
library(ggplot2)
library(randomForest)
library(rpart)
library(rpart.plot)

OUTPUT_DIR <- "/content/churn_guard_report2_outputs"
BUNDLE_FILE <- file.path(OUTPUT_DIR, "20_ChurnGuard_Model_Bundle.rds")
TRAIN_FILE <- file.path(OUTPUT_DIR, "05_training_data_prepared.csv")
METRICS_FILE <- file.path(OUTPUT_DIR, "12_model_evaluation_metrics.csv")
IMPORTANCE_FILE <- file.path(OUTPUT_DIR, "11_random_forest_feature_importance.csv")

if (!file.exists(BUNDLE_FILE)) stop("Phase 2 model bundle is missing.")
if (!file.exists(TRAIN_FILE)) stop("Prepared training data is missing.")

bundle <- readRDS(BUNDLE_FILE)
training <- read.csv(TRAIN_FILE, stringsAsFactors = FALSE, check.names = FALSE)

predictors <- bundle$predictor_names
numeric_predictors <- bundle$numeric_predictors
categorical_predictors <- bundle$categorical_predictors
factor_levels <- bundle$training_factor_levels

if (is.null(predictors) || !length(predictors)) {
  stop("Predictor metadata is missing from the Phase 2 model bundle.")
}

if (!all(predictors %in% names(training))) {
  stop("The prepared training data does not match the model metadata.")
}

# Training-derived fallbacks for any missing input values.
numeric_fill <- lapply(numeric_predictors, function(nm) {
  x <- suppressWarnings(as.numeric(training[[nm]]))
  x <- x[is.finite(x)]
  if (!length(x)) 0 else median(x)
})
names(numeric_fill) <- numeric_predictors

mode_value <- function(x) {
  x <- as.character(x)
  x <- trimws(x)
  x <- x[!is.na(x) & x != ""]
  if (!length(x)) return("Unknown")
  u <- unique(x)
  u[which.max(tabulate(match(x, u)))]
}

categorical_fill <- lapply(categorical_predictors, function(nm) {
  mode_value(training[[nm]])
})
names(categorical_fill) <- categorical_predictors

if (is.null(factor_levels)) factor_levels <- list()

for (nm in categorical_predictors) {
  if (is.null(factor_levels[[nm]]) || !length(factor_levels[[nm]])) {
    factor_levels[[nm]] <- sort(unique(as.character(training[[nm]])))
  }
  if (!("Other" %in% factor_levels[[nm]])) {
    factor_levels[[nm]] <- c(factor_levels[[nm]], "Other")
  }
}

# Same feature order, types and factor levels used by the fitted models.
prepare_new_data <- function(input_data) {
  names(input_data) <- tolower(
    gsub("[^A-Za-z0-9]+", "_", names(input_data))
  )

  missing_columns <- setdiff(predictors, names(input_data))

  if (length(missing_columns)) {
    stop(
      paste(
        "Required predictor columns missing:",
        paste(missing_columns, collapse = ", ")
      )
    )
  }

  x <- input_data[, predictors, drop = FALSE]

  if (all(c("tenure", "totalcharges") %in% names(x))) {
    x$tenure <- suppressWarnings(as.numeric(as.character(x$tenure)))
    x$totalcharges <- suppressWarnings(as.numeric(as.character(x$totalcharges)))
    x$totalcharges[
      is.na(x$totalcharges) & !is.na(x$tenure) & x$tenure == 0
    ] <- 0
  }

  for (nm in numeric_predictors) {
    x[[nm]] <- suppressWarnings(as.numeric(as.character(x[[nm]])))
    x[[nm]][!is.finite(x[[nm]])] <- NA_real_
    x[[nm]][is.na(x[[nm]])] <- numeric_fill[[nm]]
  }

  for (nm in categorical_predictors) {
    x[[nm]] <- trimws(as.character(x[[nm]]))
    x[[nm]][is.na(x[[nm]]) | x[[nm]] == ""] <- categorical_fill[[nm]]

    lv <- factor_levels[[nm]]
    x[[nm]][!x[[nm]] %in% lv] <- "Other"
    x[[nm]] <- factor(x[[nm]], levels = lv)
  }

  x
}

predict_churn_probability <- function(input_data, model_name) {
  x <- prepare_new_data(input_data)

  if (model_name == "Logistic Regression") {
    p <- predict(bundle$logistic_regression, newdata = x, type = "response")
  } else if (model_name == "Decision Tree") {
    pm <- predict(bundle$decision_tree, newdata = x, type = "prob")
    p <- if ("Yes" %in% colnames(pm)) pm[, "Yes"] else rep(0, nrow(x))
  } else if (model_name == "Random Forest") {
    pm <- predict(bundle$random_forest, newdata = x, type = "prob")
    p <- if ("Yes" %in% colnames(pm)) pm[, "Yes"] else rep(0, nrow(x))
  } else {
    stop("Unsupported model.")
  }

  pmin(pmax(as.numeric(p), 0), 1)
}

risk_label <- function(p) {
  ifelse(p >= 0.60, "High",
         ifelse(p >= 0.30, "Medium", "Low"))
}

model_choices <- c(
  "Random Forest",
  "Logistic Regression",
  "Decision Tree"
)

# Demo defaults from actual training rows.
# Used only to pre-populate the input form.
demo_row <- training[1, predictors, drop = FALSE]

ui <- fluidPage(
  tags$head(
    tags$style(HTML("
      @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
      body {
        background: #f8fafc;
        color: #1e293b;
        font-family: 'Inter', sans-serif;
        padding: 20px 40px;
      }
      .navbar-brand {
        font-weight: 700;
        color: #0f172a;
      }
      .well {
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 16px;
        box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.05), 0 2px 4px -2px rgb(0 0 0 / 0.05);
        padding: 24px;
      }
      .metric-card {
        background: #ffffff;
        border: 1px solid #e2e8f0;
        border-radius: 16px;
        padding: 24px;
        margin-bottom: 20px;
        box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.05), 0 2px 4px -2px rgb(0 0 0 / 0.05);
        transition: transform 0.2s ease-in-out;
      }
      .metric-card:hover {
        transform: translateY(-2px);
      }
      .metric-value {
        font-size: 36px;
        font-weight: 800;
        color: #4f46e5;
        letter-spacing: -0.05em;
      }
      .muted {
        color: #64748b;
        font-size: 14px;
      }
      h2, h3, h4 {
        color: #0f172a;
        font-weight: 700;
        letter-spacing: -0.02em;
      }
      .btn-primary {
        background: #4f46e5;
        border-color: #4f46e5;
        border-radius: 10px;
        font-weight: 600;
        padding: 10px 20px;
        transition: all 0.2s;
      }
      .btn-primary:hover {
        background: #4338ca;
        border-color: #4338ca;
        transform: translateY(-1px);
      }
      .nav-tabs {
        border-bottom: 2px solid #e2e8f0;
        margin-bottom: 24px;
      }
      .nav-tabs > li > a {
        font-weight: 500;
        color: #64748b;
        border: none;
        padding: 12px 20px;
        transition: all 0.2s;
      }
      .nav-tabs > li.active > a, .nav-tabs > li.active > a:hover, .nav-tabs > li.active > a:focus {
        color: #4f46e5;
        border: none;
        border-bottom: 3px solid #4f46e5;
        background: transparent;
        font-weight: 600;
      }
      .table {
        border-radius: 12px;
        overflow: hidden;
        border-style: hidden;
        box-shadow: 0 0 0 1px #e2e8f0;
      }
      .table thead {
        background: #f1f5f9;
      }
      .table thead th {
        font-weight: 600 !important;
        color: #334155 !important;
        border-bottom: 1px solid #e2e8f0 !important;
      }
      /* Customize Form Components */
      .form-control {
        border-radius: 10px;
        border: 1px solid #cbd5e1;
        padding: 8px 12px;
        transition: all 0.2s;
      }
      .form-control:focus {
        border-color: #818cf8;
        box-shadow: 0 0 0 3px rgba(129, 140, 248, 0.2);
      }
    "))
  ),

  div(
    style = "margin-bottom: 30px; display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #e2e8f0; padding-bottom: 20px;",
    div(
      h2("ChurnGuard Intelligence", style = "margin: 0; font-size: 32px; font-weight: 800; color: #0f172a;"),
      p("Interactive decision intelligence platform for customer retention analytics", class = "muted", style = "margin: 5px 0 0 0;")
    ),
    span("Phase 3 Active", style = "background: #ecfdf5; color: #065f46; padding: 6px 16px; border-radius: 9999px; font-weight: 600; font-size: 13px; border: 1px solid #a7f3d0;")
  ),

  sidebarLayout(
    sidebarPanel(
      width = 3,

      selectInput(
        "model",
        "Prediction Model",
        choices = model_choices,
        selected = "Random Forest"
      ),

      br(),

      sliderInput(
        "threshold",
        "Decision Threshold",
        min = 0.10,
        max = 0.90,
        value = 0.50,
        step = 0.05
      ),

      p("Calibrate classification boundaries to balance precision and recall targets.", class = "muted", style = "margin-top: 5px;"),

      hr(style = "border-color: #e2e8f0;"),

      tags$b("Data Mining Pipeline", style = "color: #0f172a; font-weight: 600;"),
      tags$ol(
        style = "margin-top: 10px; padding-left: 20px; color: #475569; line-height: 1.8;",
        tags$li("Clean & Process (Phase 1)"),
        tags$li("Feature Engineering (Phase 1)"),
        tags$li("Stratified Train/Test (Phase 2)"),
        tags$li("Ensemble Fit (Phase 2)"),
        tags$li("Interactive Scoring (Phase 3)")
      ),

      hr(style = "border-color: #e2e8f0;"),

      p(
        "Academic evaluation portal.",
        class = "muted",
        style = "font-style: italic;"
      )
    ),

    mainPanel(
      width = 9,

      tabsetPanel(
        id = "main_tabs",

        tabPanel(
          "Performance Dashboard",
          br(),

          fluidRow(
            column(
              4,
              div(
                class = "metric-card",
                div(class = "muted", style = "font-weight: 500;", "Prepared Training Rows"),
                div(class = "metric-value", style = "color: #4f46e5;", textOutput("training_count", inline = TRUE))
              )
            ),
            column(
              4,
              div(
                class = "metric-card",
                div(class = "muted", style = "font-weight: 500;", "Operational Framework"),
                h4(textOutput("model_label", inline = TRUE), style = "margin-top: 15px; color: #0f172a;")
              )
            ),
            column(
              4,
              div(
                class = "metric-card",
                div(class = "muted", style = "font-weight: 500;", "Configured Threshold"),
                h4(textOutput("threshold_label", inline = TRUE), style = "margin-top: 15px; color: #0f172a;")
              )
            )
          ),

          div(
            class = "metric-card",
            h3("Phase 2 Model Evaluation Archive", style = "margin-top: 0; margin-bottom: 8px;"),
            p(
              "Comprehensive statistical summary computed on held-out test data sets.",
              class = "muted", style = "margin-bottom: 20px;"
            ),
            div(style = "overflow-x: auto;", tableOutput("metrics_table"))
          ),

          div(
            class = "metric-card",
            h3("Feature Importance Diagnostics", style = "margin-top: 0; margin-bottom: 20px;"),
            plotOutput("importance_plot", height = 380)
          )
        ),

        tabPanel(
          "Individual Predictor",
          br(),

          p(
            "Construct and test a target customer profile below to run interactive classification inferences in real time.",
            style = "margin-bottom: 25px; color: #475569;"
          ),

          div(
            class = "metric-card",
            h3("Customer Demographics & Profile Details", style = "margin-top: 0; margin-bottom: 24px;"),
            uiOutput("customer_form"),
            br(),
            actionButton(
              "predict_one",
              "Calculate Churn Score",
              class = "btn-primary",
              style = "width: 100%; font-size: 16px; padding: 12px;"
            )
          ),

          uiOutput("single_result"),

          uiOutput("single_profile_card")
        ),

        tabPanel(
          "Batch Scoring Process",
          br(),

          p(
            "Evaluate bulk customer portfolios instantly by uploading formatted CSV data.",
            style = "color: #475569;"
          ),

          div(
            class = "metric-card",
            h3("Get Started with Batch Processing", style = "margin-top: 0;"),
            p("1. Download the baseline structure scheme to prepare your CSV columns correctly:", class = "muted"),
            downloadButton("download_template", "Download Input Template (.CSV)", class = "btn", style = "background: #f1f5f9; border: 1px solid #cbd5e1; color: #334155; font-weight: 500; border-radius: 10px;"),
            br(), br(),
            p("2. Upload your completed customer file to generate instant predictions:", class = "muted"),
            fileInput(
              "batch_file",
              NULL,
              accept = ".csv",
              width = "100%"
            ),
            actionButton(
              "score_batch",
              "Begin Scoring Sequence",
              class = "btn-primary",
              style = "width: 100%;"
            )
          ),

          uiOutput("batch_status"),
          uiOutput("batch_preview_card"),

          br()
        ),

        tabPanel(
          "Visual Diagnostics",
          br(),

          div(
            class = "metric-card",
            h3("Visual risk and probability distribution profiles", style = "margin-top: 0; margin-bottom: 8px;"),
            p("Insights derived from live batch operations or baseline model validation samples.", class = "muted", style = "margin-bottom: 30px;"),

            h4("Predicted Risk Category Share", style = "margin-top: 0; margin-bottom: 15px; color: #334155;"),
            plotOutput("risk_plot", height = 350),
            br(), hr(style = "border-color: #f1f5f9;"), br(),
            h4("Prediction Probability Densities", style = "margin-top: 0; margin-bottom: 15px; color: #334155;"),
            plotOutput("probability_plot", height = 350)
          )
        ),

        tabPanel(
          "Documentation & Architecture",
          br(),

          div(
            class = "metric-card",
            h3("Theoretical Underpinnings", style = "margin-top: 0;"),
            p("Customer attrition models operate on mathematical estimates of tenure longevity relative to billing commitments. ChurnGuard aggregates high-dimensionality user telemetry (payment habits, service configurations, billing records) into actionable retention pathways."),

            h4("Core Algorithms Embedded", style = "color: #4f46e5; margin-top: 24px;"),
            tags$ul(
              style = "line-height: 1.8; color: #475569;",
              tags$li(tags$strong("Logistic Regression:"), " Serves as our parametric probabilistic benchmark, providing direct mathematical odds ratios."),
              tags$li(tags$strong("Decision Tree (CART):"), " Generates explicit split criteria for clear logical paths."),
              tags$li(tags$strong("Random Forest:"), " Implements bootstrap aggregation (bagging) over decorrelated trees to reduce variance.")
            ),

            h4("Performance Validation Framework", style = "color: #4f46e5; margin-top: 24px;"),
            p("We leverage the held-out test subset established during Phase 2 training to preserve objective verification metrics. The metrics report precision, recall, accuracy, specificity, and Area under the ROC Curve (AUC).")
          )
        )
      )
    )
  )
)

server <- function(input, output, session) {

  output$training_count <- renderText(
    format(nrow(training), big.mark = ",")
  )

  output$model_label <- renderText(input$model)

  output$threshold_label <- renderText(
    paste0(round(input$threshold * 100), "%")
  )

  output$metrics_table <- renderTable({
    if (!file.exists(METRICS_FILE)) {
      return(data.frame(Status = "Phase 2 metrics file not found."))
    }

    m <- read.csv(METRICS_FILE, stringsAsFactors = FALSE)

    wanted <- c(
      "model", "threshold", "accuracy", "precision", "recall",
      "specificity", "f1_score", "balanced_accuracy",
      "roc_auc", "brier_score"
    )

    m <- m[, intersect(wanted, names(m)), drop = FALSE]

    numeric_cols <- names(m)[vapply(m, is.numeric, logical(1))]
    m[numeric_cols] <- lapply(m[numeric_cols], round, digits = 4)

    m
  }, striped = TRUE, bordered = TRUE, spacing = "s")

  output$importance_plot <- renderPlot({
    validate(need(file.exists(IMPORTANCE_FILE), "Feature-importance file not found."))

    imp <- read.csv(IMPORTANCE_FILE, stringsAsFactors = FALSE)

    validate(need("feature" %in% names(imp), "Feature column is missing."))

    metric <- if ("MeanDecreaseGini" %in% names(imp)) {
      "MeanDecreaseGini"
    } else {
      setdiff(names(imp), "feature")[1]
    }

    validate(need(length(metric) > 0 && !is.na(metric), "Importance values unavailable."))

    imp <- imp[order(imp[[metric]], decreasing = TRUE), , drop = FALSE]
    imp <- head(imp, 15)

    ggplot(
      imp,
      aes(x = reorder(feature, .data[[metric]]), y = .data[[metric]])
    ) +
      geom_col(fill = "#4f46e5", width = 0.6) +
      coord_flip() +
      labs(
        title = NULL,
        x = NULL,
        y = metric
      ) +
      theme_minimal(base_size = 12) +
      theme(
        panel.grid.minor = element_blank(),
        panel.grid.major.y = element_blank(),
        axis.text = element_text(color = "#475569", font = "Inter")
      )
  })

  output$customer_form <- renderUI({
    # Display fields in a 3-column responsive grid layout
    fluidRow(
      lapply(predictors, function(nm) {
        label <- tools::toTitleCase(gsub("_", " ", nm))
        input_id <- paste0("customer_", nm)

        column(
          4,
          style = "margin-bottom: 18px;",
          if (nm %in% categorical_predictors) {
            vals <- factor_levels[[nm]]
            vals <- setdiff(vals, "Other")

            default <- as.character(demo_row[[nm]][1])

            if (is.null(default) || is.na(default) || !default %in% vals) {
              default <- if (length(vals)) vals[1] else "Unknown"
            }

            selectInput(
              input_id,
              label,
              choices = vals,
              selected = default,
              width = "100%"
            )
          } else {
            value <- suppressWarnings(as.numeric(demo_row[[nm]][1]))

            if (!length(value) || !is.finite(value)) {
              value = numeric_fill[[nm]]
            }

            numericInput(
              input_id,
              label,
              value = value,
              width = "100%"
            )
          }
        )
      })
    )
  })

  single_input <- reactive({
    values <- lapply(predictors, function(nm) {
      input[[paste0("customer_", nm)]]
    })

    names(values) <- predictors

    as.data.frame(values, stringsAsFactors = FALSE, check.names = FALSE)
  })

  single_prediction <- eventReactive(input$predict_one, {
    x <- single_input()

    validate(
      need(
        all(vapply(x, function(z) length(z) > 0 && !is.null(z), logical(1))),
        "Complete all customer inputs."
      )
    )

    p <- predict_churn_probability(x, input$model)

    list(
      profile = x,
      probability = p,
      predicted_class = ifelse(p >= input$threshold, "Yes", "No"),
      risk = risk_label(p)
    )
  }, ignoreInit = TRUE)

  output$single_result <- renderUI({
    res <- single_prediction()
    p <- res$probability

    bg_class <- if (p >= 0.60) {
      "background: #fef2f2; border: 1px solid #fee2e2; color: #991b1b;"
    } else if (p >= 0.30) {
      "background: #fffbeb; border: 1px solid #fef3c7; color: #92400e;"
    } else {
      "background: #f0fdf4; border: 1px solid #dcfce7; color: #166534;"
    }

    div(
      class = "metric-card",
      style = bg_class,
      h3("Assessment Output Summary", style = "margin-top: 0; color: inherit;"),
      fluidRow(
        column(
          4,
          div(style = "font-size: 54px; font-weight: 800; line-height: 1;", paste0(round(p * 100, 1), "%")),
          div(style = "margin-top: 8px; font-weight: 500; opacity: 0.8;", "Estimated Probability")
        ),
        column(
          8,
          div(style = "font-size: 18px; font-weight: 700; margin-bottom: 5px;", paste("Action Flag:", res$predicted_class)),
          div(style = "font-size: 15px; font-weight: 500; margin-bottom: 5px;", paste("Risk Designation:", res$risk)),
          div(style = "font-size: 13px; opacity: 0.8;", paste("Evaluated via", input$model, "at", round(input$threshold * 100), "% threshold."))
        )
      )
    )
  })

  output$single_profile <- renderTable({
    res <- single_prediction()

    data.frame(
      Feature = names(res$profile),
      Value = vapply(
        res$profile,
        function(z) as.character(z[1]),
        character(1)
      )
    )
  }, striped = TRUE, bordered = TRUE)

  output$single_profile_card <- renderUI({
    req(single_prediction())
    div(
      class = "metric-card",
      h4("Evaluated Profile Signature", style = "margin-top: 0; margin-bottom: 15px;"),
      tableOutput("single_profile")
    )
  })

  output$download_template <- downloadHandler(
    filename = function() "churnguard_input_template.csv",
    content = function(file) {
      template <- as.data.frame(
        lapply(predictors, function(nm) {
          if (nm %in% numeric_predictors) {
            numeric_fill[[nm]]
          } else {
            categorical_fill[[nm]]
          }
        }),
        stringsAsFactors = FALSE,
        check.names = FALSE
      )

      names(template) <- predictors
      write.csv(template, file, row.names = FALSE)
    }
  )

  batch_results <- eventReactive(input$score_batch, {
    req(input$batch_file)

    batch <- read.csv(
      input$batch_file$datapath,
      stringsAsFactors = FALSE,
      check.names = FALSE
    )

    missing_cols <- setdiff(
      predictors,
      tolower(gsub("[^A-Za-z0-9]+", "_", names(batch)))
    )

    validate(
      need(
        !length(missing_cols),
        paste("Missing required columns:", paste(missing_cols, collapse = ", "))
      )
    )

    p <- predict_churn_probability(batch, input$model)

    result <- batch
    result$predicted_churn_probability <- p
    result$predicted_churn <- ifelse(p >= input$threshold, "Yes", "No")
    result$risk_category <- risk_label(p)
    result$scoring_model <- input$model
    result$classification_threshold <- input$threshold

    result
  }, ignoreInit = TRUE)

  output$batch_status <- renderUI({
    res <- batch_results()

    div(
      class = "metric-card",
      style = "background: #f8fafc; border: 1px solid #cbd5e1;",
      h3("Batch Processing Success", style = "margin-top: 0; color: #0f172a;"),
      fluidRow(
        column(4, div(style = "font-size: 28px; font-weight: 800; color: #4f46e5;", nrow(res)), div(class = "muted", "Total Evaluated")),
        column(4, div(style = "font-size: 28px; font-weight: 800; color: #ef4444;", sum(res$predicted_churn == "Yes")), div(class = "muted", "High Probability Churn")),
        column(4, div(style = "font-size: 28px; font-weight: 800; color: #10b981;", sum(res$predicted_churn == "No")), div(class = "muted", "Predicted Stay"))
      ),
      br(),
      downloadButton(
        "download_batch",
        "Download Fully Scored Registry (.CSV)",
        class = "btn-primary",
        style = "display: inline-block; width: auto;"
      )
    )
  })

  output$batch_preview <- renderTable({
    head(batch_results(), 10)
  }, striped = TRUE, bordered = TRUE)

  output$batch_preview_card <- renderUI({
    req(batch_results())
    div(
      class = "metric-card",
      h4("Scored Preview (Top 10 Records)", style = "margin-top: 0; margin-bottom: 15px;"),
      div(style = "overflow-x: auto;", tableOutput("batch_preview"))
    )
  })

  output$download_batch <- downloadHandler(
    filename = function() {
      paste0("churnguard_scored_customers_", Sys.Date(), ".csv")
    },
    content = function(file) {
      write.csv(batch_results(), file, row.names = FALSE)
    }
  )

  # Use batch predictions when available; otherwise sample prepared training rows.
  visual_data <- reactive({
    if (!is.null(input$score_batch) && input$score_batch > 0) {
      tryCatch(batch_results(), error = function(e) NULL)
    } else {
      NULL
    }
  })

  output$risk_plot <- renderPlot({
    d <- visual_data()

    if (is.null(d)) {
      validate(need("churn" %in% names(training), "Training target unavailable."))

      plot_data <- data.frame(
        Category = as.character(training$churn)
      )

      ggplot(plot_data, aes(x = Category, fill = Category)) +
        geom_bar(width = 0.5, show.legend = FALSE) +
        scale_fill_manual(values = c("No" = "#10b981", "Yes" = "#ef4444")) +
        labs(
          x = "Observed Churn Class",
          y = "Customer Count"
        ) +
        theme_minimal(base_size = 12) +
        theme(
          text = element_text(family = "Inter", color = "#475569"),
          panel.grid.minor = element_blank(),
          panel.grid.major.x = element_blank()
        )
    } else {
      ggplot(d, aes(x = risk_category, fill = risk_category)) +
        geom_bar(width = 0.5, show.legend = FALSE) +
        scale_fill_manual(values = c("Low" = "#10b981", "Medium" = "#f59e0b", "High" = "#ef4444")) +
        labs(
          x = "Risk Category",
          y = "Customer Count"
        ) +
        theme_minimal(base_size = 12) +
        theme(
          text = element_text(family = "Inter", color = "#475569"),
          panel.grid.minor = element_blank(),
          panel.grid.major.x = element_blank()
        )
    }
  })

  output$probability_plot <- renderPlot({
    d <- visual_data()

    if (is.null(d)) {
      set.seed(42)
      sample_n <- min(300, nrow(training))
      sample_data <- training[
        sample(seq_len(nrow(training)), sample_n),
        predictors,
        drop = FALSE
      ]

      p <- predict_churn_probability(sample_data, input$model)

      plot_data <- data.frame(probability = p)

      ggplot(plot_data, aes(x = probability)) +
        geom_histogram(
          bins = 20,
          fill = "#4f46e5",
          colour = "white",
          alpha = 0.8
        ) +
        labs(
          x = "Predicted Churn Probability",
          y = "Customer Count"
        ) +
        theme_minimal(base_size = 12) +
        theme(
          text = element_text(family = "Inter", color = "#475569"),
          panel.grid.minor = element_blank()
        )
    } else {
      ggplot(
        d,
        aes(x = predicted_churn_probability)
      ) +
        geom_histogram(
          bins = 20,
          fill = "#4f46e5",
          colour = "white",
          alpha = 0.8
        ) +
        labs(
          x = "Predicted Churn Probability",
          y = "Customer Count"
        ) +
        theme_minimal(base_size = 12) +
        theme(
          text = element_text(family = "Inter", color = "#475569"),
          panel.grid.minor = element_blank()
        )
    }
  })
}

shinyApp(ui = ui, server = server)
'''

app_file = APP_DIR / "app.R"
app_file.write_text(app_code, encoding="utf-8")

# ------------------------------------------------
# 4. Install Shiny dependencies and check app syntax
# ------------------------------------------------

setup_code = r'''
needed <- c("shiny", "ggplot2", "randomForest", "rpart", "rpart.plot")
missing <- needed[!vapply(needed, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}
parse(file = "/content/churn_guard_phase3/app.R")
cat("App syntax check passed.\n")
'''

setup = subprocess.run(
    [rscript, "-e", setup_code],
    text=True,
    capture_output=True,
    timeout=600
)

if setup.returncode != 0:
    print(setup.stdout)
    print(setup.stderr)
    raise RuntimeError("R dependency installation or app syntax check failed.")

print(setup.stdout)

# ------------------------------------------------
# 5. Launch Shiny on a Colab-accessible port
# ------------------------------------------------

PORT = 3838
log_file = APP_DIR / "shiny.log"
err_file = APP_DIR / "shiny_error.log"

# Stop an earlier process on this port if there is one.
subprocess.run(
    ["bash", "-lc", f"fuser -k {PORT}/tcp >/dev/null 2>&1 || true"],
    capture_output=True
)

log_handle = open(log_file, "w")
err_handle = open(err_file, "w")

process = subprocess.Popen(
    [
        rscript,
        "-e",
        (
            "shiny::runApp("
            "'/content/churn_guard_phase3', "
            "host='0.0.0.0', "
            f"port={PORT}, "
            "launch.browser=FALSE)"
        )
    ],
    stdout=log_handle,
    stderr=err_handle,
    start_new_session=True
)

# Wait briefly for the server to start.
started = False

for _ in range(30):
    time.sleep(1)

    if process.poll() is not None:
        break

    try:
        import socket
        with socket.create_connection(("127.0.0.1", PORT), timeout=1):
            started = True
            break
    except OSError:
        pass

if not started:
    print("App startup log:")
    print(log_file.read_text(errors="ignore")[-5000:] if log_file.exists() else "")
    print("Error log:")
    print(err_file.read_text(errors="ignore")[-5000:] if err_file.exists() else "")
    raise RuntimeError("Shiny did not start successfully. See the logs above.")

print("ChurnGuard Phase 3 started on port", PORT)

# ------------------------------------------------
# 6. Display inside Google Colab
# ------------------------------------------------

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=1100)
except Exception as exc:
    print("Could not automatically embed the app:", exc)
    print("Try running this in a separate Python cell:")
    print("from google.colab import output")
    print(f"output.serve_kernel_port_as_iframe({PORT}, height=1100)")

print("Phase 3 app file:", app_file)
print("Shiny log:", log_file)
print("Shiny error log:", err_file)
print("Keep the Colab runtime alive while demonstrating the app.")


expression(options(shiny.maxRequestSize = 30 * 1024^2), required_packages <- c("shiny", 
    "ggplot2", "randomForest", "rpart", "rpart.plot"), missing_packages <- required_packages[!vapply(required_packages, 
    requireNamespace, logical(1), quietly = TRUE)], if (length(missing_packages)) {
    install.packages(missing_packages, repos = "https://cloud.r-project.org")
}, library(shiny), library(ggplot2), library(randomForest), library(rpart), 
    library(rpart.plot), OUTPUT_DIR <- "/content/churn_guard_report2_outputs", 
    BUNDLE_FILE <- file.path(OUTPUT_DIR, "20_ChurnGuard_Model_Bundle.rds"), 
    TRAIN_FILE <- file.path(OUTPUT_DIR, "05_training_data_prepared.csv"), 
    METRICS_FILE <- file.path(OUTPUT_DIR, "12_model_evaluation_metrics.csv"), 
    IMPORTANCE_FILE <- file.path(OUTPUT_DIR, "11_random_forest_feature_importance.csv"), 
    if (!file.exists(BUNDLE_FILE)) stop("Phase 2 model bundle is missing."), 
    if (!file.exists(TRAIN_FILE)) stop("Prepared training data is missin

<IPython.core.display.Javascript object>

Phase 3 app file: /content/churn_guard_phase3/app.R
Shiny log: /content/churn_guard_phase3/shiny.log
Shiny error log: /content/churn_guard_phase3/shiny_error.log
Keep the Colab runtime alive while demonstrating the app.
